# MedGemma 4B on VQA-RAD: Medical Visual Question Answering on a free Colab GPU

This notebook evaluates Google's **MedGemma 4B (instruction-tuned)** vision-language model, zero-shot, on the **VQA-RAD** test set (radiology images with clinician-written questions). It follows the evaluation in the MedGemma Technical Report (Sellergren et al., 2025):

* **Closed (yes/no) questions:** accuracy, precision, recall, F1
* **Open-ended questions:** tokenized F1 (the report's metric), exact match

It runs on a **free Colab T4 GPU** by loading the model in 4-bit (NF4) precision.

### Before you click *Run all*
1. **Runtime → Change runtime type → T4 GPU**.
2. Sign in at https://huggingface.co, open https://huggingface.co/google/medgemma-4b-it and **accept the license** (it's instant).
3. Create a **Read** token at https://huggingface.co/settings/tokens. Either add it as a Colab secret named `HF_TOKEN` (key icon in the left bar), or paste it when the login box appears.
Outputs saved: `predictions.jsonl` (one JSON record per question), `metrics.json`, `confusion_matrix.png`, `sample_predictions.png`.

Expected run time: about 30–50 minutes in total (mostly model download and inference).

In [ ]:
# 1. Install libraries
!pip install -q -U "transformers>=4.50.0" accelerate bitsandbytes datasets scikit-learn

In [ ]:
# 2. Log in to Hugging Face (needed because MedGemma is a gated model)
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in with Colab secret HF_TOKEN")
except Exception:
    login()  # shows a box where you paste your token

In [ ]:
# 3. Load MedGemma 4B in 4-bit
import torch, time
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"   # MedGemma 1 (matches the technical report). Alternative: "google/medgemma-1.5-4b-it"
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU, then rerun."

# T4 (compute capability 7.5) has no native bfloat16, so use float32 compute there for numerical safety
major, _ = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float32
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| compute dtype:", DTYPE)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=DTYPE,
                                                    device_map="auto", quantization_config=bnb)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Model loaded in {time.time()-t0:.0f}s | GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# 4. Load the VQA-RAD test set
from datasets import load_dataset
ds = load_dataset("flaviagiammarino/vqa-rad", split="test")

MAX_QUESTIONS = None   # set e.g. 100 for a quick run; None = whole test set
if MAX_QUESTIONS:
    ds = ds.select(range(min(MAX_QUESTIONS, len(ds))))

def is_closed(ans):
    return ans.strip().lower() in ("yes", "no")

n_closed = sum(is_closed(a) for a in ds["answer"])
print(f"Test questions: {len(ds)} | closed (yes/no): {n_closed} | open-ended: {len(ds)-n_closed}")
ds[0]["image"]

In [ ]:
# 5. Prompt templates (same as slide 13)
SYSTEM_PROMPT_DETAILED = '''You are an expert radiologist assisting with visual
question answering on radiology images (X-ray, CT, MRI).
Look carefully at the given image and answer the clinician's question.
Rules:
 1. Base your answer only on what is visible in the image.
 2. For yes/no questions reply with exactly one word: yes or no.
 3. For other questions reply with the shortest correct phrase
    (1-5 words) and give no explanation.
 4. Use standard anatomical terms (e.g. 'left lung', 'right kidney').
 5. If you are unsure, still give your single best answer.'''

# Token-optimised version used in the final pipeline
SYSTEM_PROMPT = "You are an expert radiologist."
CLOSED_SUFFIX = "Answer yes or no."
OPEN_SUFFIX   = "Answer in as few words as possible."

def build_messages(image, question, closed, system_prompt=SYSTEM_PROMPT):
    suffix = CLOSED_SUFFIX if closed else OPEN_SUFFIX
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": f"{question}\n{suffix}"}]},
    ]

def encode(messages):
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                         return_dict=True, return_tensors="pt").to(model.device, dtype=DTYPE)

In [ ]:
# 6. Token optimisation: compare input tokens of detailed vs optimised prompt
ex0 = ds[0]
img0 = ex0["image"].convert("RGB")
n_detailed = encode(build_messages(img0, ex0["question"], True, SYSTEM_PROMPT_DETAILED))["input_ids"].shape[-1]
n_opt      = encode(build_messages(img0, ex0["question"], True, SYSTEM_PROMPT))["input_ids"].shape[-1]
print(f"Input tokens with detailed prompt : {n_detailed}")
print(f"Input tokens with optimised prompt: {n_opt}")
print(f"Reduction: {100*(n_detailed-n_opt)/n_detailed:.1f}% fewer input tokens per question")

In [ ]:
# 7. Run MedGemma on every test question and save JSON records
import json, pandas as pd
from tqdm.auto import tqdm

def ask(image, question, closed):
    inputs = encode(build_messages(image, question, closed))
    n_in = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=8 if closed else 24, do_sample=False)
    gen = out[0][n_in:]
    return processor.decode(gen, skip_special_tokens=True).strip(), n_in, len(gen)

torch.cuda.reset_peak_memory_stats()
rows, t_start = [], time.time()
with open("predictions.jsonl", "w") as f:
    for i, ex in enumerate(tqdm(ds)):
        img = ex["image"].convert("RGB")
        closed = is_closed(ex["answer"])
        t = time.time()
        try:
            pred, n_in, n_out = ask(img, ex["question"], closed)
        except Exception as e:
            print("Error on", i, e); pred, n_in, n_out = "", 0, 0
        rec = {"id": i, "type": "closed" if closed else "open", "question": ex["question"],
               "raw_output": pred, "ground_truth": ex["answer"],
               "input_tokens": int(n_in), "output_tokens": int(n_out), "seconds": round(time.time() - t, 3)}
        f.write(json.dumps(rec) + "\n")
        rows.append({"idx": i, "question": ex["question"], "answer": ex["answer"], "prediction": pred,
                     "type": rec["type"], "seconds": rec["seconds"],
                     "input_tokens": rec["input_tokens"], "output_tokens": rec["output_tokens"]})

TOTAL_MIN = (time.time() - t_start) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1e9
df = pd.DataFrame(rows)
df.to_csv("medgemma_vqarad_predictions.csv", index=False)
print(f"Done in {TOTAL_MIN:.1f} min | peak GPU memory {PEAK_GB:.2f} GB | avg input tokens {df.input_tokens.mean():.0f}")
df.head(10)

In [ ]:
# 8. Compute metrics (same definitions as the MedGemma report)
import re, json, collections

def norm(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return " ".join(s.split())

def to_yn(pred):
    toks = norm(pred).split()
    return toks[0] if toks and toks[0] in ("yes", "no") else "invalid"

def token_f1(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    if not p or not g:
        return float(p == g)
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

cl = df[df.type == "closed"].copy()
cl["gold"] = cl.answer.str.strip().str.lower()
cl["pred_yn"] = cl.prediction.apply(to_yn)
acc = (cl.gold == cl.pred_yn).mean()
tp = ((cl.gold == "yes") & (cl.pred_yn == "yes")).sum()
fp = ((cl.gold == "no") & (cl.pred_yn == "yes")).sum()
fn = ((cl.gold == "yes") & (cl.pred_yn != "yes")).sum()
prec = tp / (tp + fp) if tp + fp else 0
rec = tp / (tp + fn) if tp + fn else 0
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
invalid = (cl.pred_yn == "invalid").sum()

op = df[df.type == "open"].copy()
op["tokf1"] = [token_f1(p, g) for p, g in zip(op.prediction, op.answer)]
op["em"] = [norm(p) == norm(g) for p, g in zip(op.prediction, op.answer)]
df.loc[cl.index, "score"] = (cl.gold == cl.pred_yn).astype(float)
df.loc[op.index, "score"] = op.tokf1
overall_tokf1 = df.score.mean()   # closed: 1/0, open: token F1 (report's "overall token F1")

pct = lambda x: f"{100*x:.1f}"
M = {
    "N_TOTAL": len(df), "N_CLOSED": len(cl), "N_OPEN": len(op),
    "CLOSED_ACC": pct(acc), "CLOSED_PREC": pct(prec), "CLOSED_REC": pct(rec), "CLOSED_F1": pct(f1),
    "INVALID": int(invalid),
    "OPEN_F1": pct(op.tokf1.mean()) if len(op) else "n/a",
    "OPEN_EM": pct(op.em.mean()) if len(op) else "n/a",
    "OVERALL_F1": pct(overall_tokf1),
    "LATENCY": f"{df.seconds.mean():.2f}", "GPU_MEM": f"{PEAK_GB:.1f}",
    "GPU_NAME": GPU_NAME.replace("Tesla ", ""), "RUNTIME_MIN": f"{TOTAL_MIN:.0f}",
}
json.dump(M, open("metrics.json", "w"), indent=2)
for k, v in M.items():
    print(f"{k:12s} {v}")

In [ ]:
# 9. Figures: confusion matrix + sample predictions
import matplotlib.pyplot as plt
import numpy as np

labels_true = ["yes", "no"]
labels_pred = ["yes", "no"] + (["invalid"] if invalid else [])
cm = np.array([[((cl.gold == t) & (cl.pred_yn == p)).sum() for p in labels_pred] for t in labels_true])
fig, ax = plt.subplots(figsize=(5, 4.2), dpi=200)
ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(len(labels_pred)), labels_pred); ax.set_yticks(range(2), labels_true)
ax.set_xlabel("MedGemma prediction"); ax.set_ylabel("Ground truth")
ax.set_title(f"VQA-RAD closed questions (n={len(cl)})")
plt.tight_layout(); plt.savefig("confusion_matrix.png", bbox_inches="tight"); plt.show()

# sample grid: 3 correct + 3 incorrect
right = df[df.score >= 0.99].head(3).idx.tolist()
wrong = df[df.score < 0.5].head(3).idx.tolist()
pick = right + wrong
fig, axes = plt.subplots(2, 3, figsize=(12, 8.2), dpi=150)
for ax, i in zip(axes.flat, pick):
    r = df.loc[i]
    ax.imshow(ds[int(i)]["image"].convert("L"), cmap="gray"); ax.axis("off")
    ok = r.score >= 0.99
    q = r.question if len(r.question) < 60 else r.question[:57] + "..."
    ax.set_title(f"Q: {q}\nTrue: {r.answer} | MedGemma: {r.prediction[:30]}",
                 fontsize=9, color="green" if ok else "red")
plt.tight_layout(); plt.savefig("sample_predictions.png", bbox_inches="tight"); plt.show()

In [ ]:
# 10. Final summary (numbers for the Results slide) + download files
print("=== MedGemma 4B (4-bit) on VQA-RAD test set ===")
print(f"Closed questions : {M['N_CLOSED']} | Closed accuracy : {M['CLOSED_ACC']}%")
print(f"Open questions   : {M['N_OPEN']} | Open token F1   : {M['OPEN_F1']}%  | Exact match: {M['OPEN_EM']}%")
print(f"Overall token F1 : {M['OVERALL_F1']}%")
print(f"Time/question    : {M['LATENCY']} s | Peak GPU memory: {M['GPU_MEM']} GB | Total: {M['RUNTIME_MIN']} min on {M['GPU_NAME']}")
try:
    from google.colab import files
    for f in ["metrics.json", "predictions.jsonl", "medgemma_vqarad_predictions.csv",
              "confusion_matrix.png", "sample_predictions.png"]:
        files.download(f)
except Exception:
    pass